<a href="https://colab.research.google.com/github/JoaoAntoni07/estudo-catijr/blob/main/semana-06/semana06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 06

## Algoritmos não supervisionados

### K-Means

#### Carregando os dados e importando as bibliotecas

In [11]:
# importando as bibliotecas necessárias

# EDA e vizualização de dados
import pandas as pd
import plotly.express as px
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# ML
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer

# Otimização
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 19.6 MB/s eta 0:00:00


In [12]:
# importando o dataset
from google.colab import drive
drive.mount('/content/drive')
path = '/content/drive/MyDrive/dataset_clientes.csv'
df_clientes = pd.read_csv(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   atividade_economica     500 non-null    object 
 1   faturamento_mensal      500 non-null    float64
 2   numero_de_funcionarios  500 non-null    int64  
 3   localizacao             500 non-null    object 
 4   idade                   500 non-null    int64  
 5   inovacao                500 non-null    int64  
dtypes: float64(1), int64(3), object(2)
memory usage: 23.6+ KB


In [ ]:
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao
0,Comércio,713109.95,12,Rio de Janeiro,6,1
1,Comércio,790714.38,9,São Paulo,15,0
2,Comércio,1197239.33,17,São Paulo,4,9
3,Indústria,449185.78,15,São Paulo,6,0
4,Agronegócio,1006373.16,15,São Paulo,15,8
5,Serviços,1629562.41,16,Rio de Janeiro,11,4
6,Serviços,771179.95,13,Vitória,0,1
7,Serviços,707837.61,16,São Paulo,10,6
8,Comércio,888983.66,17,Belo Horizonte,10,1
9,Indústria,1098512.64,13,Rio de Janeiro,9,3


#### EDA

In [ ]:
# distribuição da variável inovação
percentual_inovacao = df_clientes.value_counts('inovacao') / len(df_clientes) * 100
px.bar(percentual_inovacao, color=percentual_inovacao.index)

#### ANOVA

In [ ]:
# teste ANOVA
# verifica se há variações significativas na média de faturamento mensal para diferentes níveis de inovação
# suposições:
# -observações independentes
# -variável dependente é contínua
# -seguem uma distribuição normal
# -homogeneidade das variências
# -amostras sejam de tamanhos iguais

In [13]:
# checar se as variancias (faturamento) entre grupos (inovação) são homogêneas
# aplicar teste de bartlett
# H0 - variâncias são iguais
# H1 - variâncias não são iguais

from scipy.stats import bartlett

# separando os dados de faturamento em grupos coim base na coluna inovação
dados_agrupados = [df_clientes['faturamento_mensal'][df_clientes['inovacao'] == grupo] for grupo in df_clientes['inovacao'].unique()]

# executar o teste de bartlett
bartlett_test_statistic, barlett_p_value = bartlett(*dados_agrupados)

# exibindo resultados
print(f'Estatística de Bartlett: {bartlett_test_statistic}')
print(f'P-Value de Bartlett: {barlett_p_value}')

Estatística de Bartlett: 10.901203117231173
P-Value de Bartlett: 0.28254182954905804


In [14]:
# executar o teste de Shapiro-Wilk
# verificar se os dados seguem uma distribuição normal

from scipy.stats import shapiro

# executar o teste
shapiro_test_statistic, shapiro_p_value = shapiro(df_clientes['faturamento_mensal'])

# exibindo resultados
print(f'Estatística de SW: {shapiro_test_statistic}')
print(f'P-Value de SW: {shapiro_p_value}')

Estatística de SW: 0.9959857602472711
P-Value de SW: 0.23513451034389005


In [15]:
# aplicar a ANOVA de Welch, pois as amostras são de tamanhos diferentes
# H0 - não há diferenças significativas entre as médias dos grupos
# H1 - há pelo menos uma difderença significativa entre as médias dos grupos
!pip install pingouin
from pingouin import welch_anova

aov = welch_anova(dv='faturamento_mensal', between='inovacao', data=df_clientes)

# Exibindo as colunas do DataFrame para diagnóstico
print("Colunas do DataFrame aov:", aov.columns)

# exibindo os resultados
print(f'Estatística do teste de ANOVA Welch: {aov.loc[0, 'F']}')
print(f'P-Value de teste de ANOVA Welch: {aov.loc[0, 'p_unc']}')

Colunas do DataFrame aov: Index(['Source', 'ddof1', 'ddof2', 'F', 'p_unc', 'np2'], dtype='object')
Estatística do teste de ANOVA Welch: 1.1269836194061693
P-Value de teste de ANOVA Welch: 0.34526211273912577


#### Treinar o algorítmo K-Means

In [16]:
# selecionar as colunas relevantes para cluster
X = df_clientes.copy()

# separar variavies numericas, categoricas e ordinais
numeric_features = ['faturamento_mensal', 'numero_de_funcionarios', 'idade']
categorical_features = ['localizacao', 'atividade_economica']
ordinal_features = ['inovacao']

# aplicar transformações por tipo
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder()
ordinal_transformer = OrdinalEncoder()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
        ('ord', ordinal_transformer, ordinal_features)
    ]
)

# transformar os dados
X_transformed = preprocessor.fit_transform(X)

In [ ]:
X_transformed

array([[-0.74634498, -0.54179191, -1.10058849, ...,  0.        ,
         0.        ,  1.        ],
       [-0.56165548, -1.5035527 ,  1.94344851, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.40582654,  1.06114274, -1.77704115, ...,  0.        ,
         0.        ,  9.        ],
       ...,
       [ 2.8196246 , -1.18296577,  0.25231684, ...,  0.        ,
         1.        ,  0.        ],
       [ 1.03321411, -0.54179191, -1.43881482, ...,  0.        ,
         0.        ,  3.        ],
       [-2.03011486, -0.22120498, -1.77704115, ...,  1.        ,
         0.        ,  9.        ]])

###E Executando o K-Means com Optuna

In [18]:
# optuna para otimização de hiperparâmetros
def kmeans_objective(trial):

  # definir os hiperparametros a serem ajustados
  n_clusters = trial.suggest_int('n_clusters', 3, 10)

  # criar o modelo
  modelo_kmeans = KMeans(n_clusters=n_clusters, random_state=51)

  # treinar o modelo
  modelo_kmeans.fit(X_transformed)

  # calculando o Silhouette Score
  silhouette_avg = silhouette_score(X_transformed, modelo_kmeans.labels_)

  return silhouette_avg

In [19]:
# criar um estudo optuna
search_space = {'n_clusters': [3, 4, 5, 6, 7, 8, 9, 10]}
sampler = optuna.samplers.GridSampler(search_space=search_space)
estudo_kmeans = optuna.create_study(direction='maximize', sampler=sampler)

#rodar o estudo
estudo_kmeans.optimize(kmeans_objective, n_trials=100)

[I 2026-07-11 18:32:29,984] A new study created in memory with name: no-name-53ba9d2a-4f86-4784-bca2-8a5e6b85f07e
[I 2026-07-11 18:32:30,135] Trial 0 finished with value: 0.12513401433552748 and parameters: {'n_clusters': 9}. Best is trial 0 with value: 0.12513401433552748.
[I 2026-07-11 18:32:30,189] Trial 1 finished with value: 0.1684229003473514 and parameters: {'n_clusters': 5}. Best is trial 1 with value: 0.1684229003473514.
[I 2026-07-11 18:32:30,226] Trial 2 finished with value: 0.16677789435822268 and parameters: {'n_clusters': 4}. Best is trial 1 with value: 0.1684229003473514.
[I 2026-07-11 18:32:30,291] Trial 3 finished with value: 0.12129203871770099 and parameters: {'n_clusters': 10}. Best is trial 1 with value: 0.1684229003473514.
[I 2026-07-11 18:32:30,337] Trial 4 finished with value: 0.15000668988973456 and parameters: {'n_clusters': 6}. Best is trial 1 with value: 0.1684229003473514.
[I 2026-07-11 18:32:30,373] Trial 5 finished with value: 0.2438038180709434 and param

#### Análise de resultados

In [25]:
# melhor configuração encontrada pelo optuna
best_params = estudo_kmeans.best_params

# instanciando o modelo K-Means com os melhores parâmetros
best_kmeans = KMeans(n_clusters=best_params['n_clusters'], random_state=51)
best_kmeans.fit(X_transformed)

# calculando o Silhouette Score
best_silhouette = silhouette_score(X_transformed, best_kmeans.labels_)

print(f'k (número de clusters): {best_params['n_clusters']}')
print(f'Silhouette Score: {best_silhouette}')

k (número de clusters): 3
Silhouette Score: 0.2438038180709434


In [26]:
# criar coluna com cluster escolhido
df_clientes['cluster'] = best_kmeans.labels_

In [28]:
df_clientes.head(10)

,atividade_economica,faturamento_mensal,numero_de_funcionarios,localizacao,idade,inovacao,cluster
0,Comércio,713109.95,12,Rio de Janeiro,6,1,0
1,Comércio,790714.38,9,São Paulo,15,0,0
2,Comércio,1197239.33,17,São Paulo,4,9,1
3,Indústria,449185.78,15,São Paulo,6,0,0
4,Agronegócio,1006373.16,15,São Paulo,15,8,1
5,Serviços,1629562.41,16,Rio de Janeiro,11,4,2
6,Serviços,771179.95,13,Vitória,0,1,0
7,Serviços,707837.61,16,São Paulo,10,6,1
8,Comércio,888983.66,17,Belo Horizonte,10,1,0
9,Indústria,1098512.64,13,Rio de Janeiro,9,3,2


#### Vizualizar os dados

In [31]:
# cruzar idade e faturamento, apresetando os clusters
px.scatter(df_clientes, x='idade', y='faturamento_mensal', color='cluster')

In [32]:
# cruzar inovaçãp e faturamento, apresetando os clusters
px.scatter(df_clientes, x='inovacao', y='faturamento_mensal', color='cluster')

#### Salvar o modelo e Pipeline

In [34]:
import joblib

# salvar o modelo
joblib.dump(best_kmeans, 'modelo_clusterizacao_clientes.pkl')

# salvar o pipeline
joblib.dump(preprocessor, 'pipeline_clusterizacao_clientes.pkl')

['pipeline_clusterizacao_clientes.pkl']

#### Aplicação Batch com o Gradio

In [37]:
import gradio as gr

modelo = joblib.load('./modelo_clusterizacao_clientes.pkl')
preprocessor = joblib.load('./pipeline_clusterizacao_clientes.pkl')

def clustering(arquivo):
  #carregar o csv em um dataframe
  df_empresas = pd.read_csv(arquivo.name)

  # transformar os dados do df para o formato que o KMeans precisa
  X_transformed = preprocessor.fit_transform(df_empresas)

  # treinar modelo
  modelo.fit(X_transformed)

  # criar a clonua cluster no df
  df_empresas['cluster'] = modelo.labels_
  df_empresas.to_csv('.clusters.csv', index=False)

  return './clusters.csv'

In [40]:
# criar a interface
app = gr.Interface(
    clustering,
    gr.File(file_types=[".csv"]),
    "file"
)

# rodar a aplicação
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f327eccd79e7da6ebc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
